# Netflix Viewing Hours and Promotion Opportunities

Which promotion opportunities should Netflix prioritize testing to increase total weekly viewing hours?

Compare content-group viewing hours, then examine older-season chart presence around new-season launches. The selected launch sample is exploratory; chart absence does not mean zero viewing.


## 1. Load the frozen weekly chart snapshot


In [1]:
from pathlib import Path
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if not (ROOT / 'data').exists():
        ROOT = ROOT.parent
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
OUT = ROOT / 'outputs' / 'tables'
OUT.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
if 'display' not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(value):
            if hasattr(value, 'data'):
                value = value.data
            if hasattr(value, 'to_string'):
                print(value.to_string(index=False))
            else:
                print(value)

import pandas as pd

data_path = ROOT / "data/all-weeks-global.tsv"
df = pd.read_csv(data_path, sep="\t")

df["week"] = pd.to_datetime(df["week"])

tv = df[
    df["category"].isin(["TV (English)", "TV (Non-English)"])
].copy()

print("Chart coverage:", tv["week"].min(), "to", tv["week"].max())
print("TV records:", len(tv))

display(tv.head())
display(tv.columns.to_list())

Chart coverage: 2021-07-04 00:00:00 to 2026-09-27 00:00:00
TV records: 5480


week,category,weekly_rank,show_title,season_title,weekly_hours_viewed,runtime,weekly_views,cumulative_weeks_in_top_10
2026-09-27,TV (English),1,Monster: The Lizzie Borden Story,Monster: The Lizzie Borden Story: Season 1,95900000,6.6333,14500000.0,2
2026-09-27,TV (English),2,Wonka's The Golden Ticket,Wonka's The Golden Ticket: Season 1,28300000,6.1167,4600000.0,1
2026-09-27,TV (English),3,Crew Girl,Crew Girl: Season 1,28100000,6.2000,4500000.0,3
2026-09-27,TV (English),4,A Different World,A Different World: Season 1,23000000,5.4500,4200000.0,1
2026-09-27,TV (English),5,The Gentlemen,The Gentlemen: Season 2,31000000,7.4333,4200000.0,4


['week', 'category', 'weekly_rank', 'show_title', 'season_title', 'weekly_hours_viewed', 'runtime', 'weekly_views', 'cumulative_weeks_in_top_10']

## Compare viewing hours across content groups

Compare TV and film hours, then English and non-English TV. Annual comparisons use complete years 2022–2025. Shares describe chart-listed hours, not the entire Netflix catalog.


In [2]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from viewing_hours import export_viewing_hours

monthly_hours, annual_hours = export_viewing_hours(df, OUT)
display(annual_hours.round(2))


category,Films (English),Films (Non-English),TV (English),TV (Non-English),film_hours_millions,tv_hours_millions,total_hours_millions,tv_share_pct,non_english_share_of_tv_pct
,8679.58,3632.79,20159.90,9946.54,12312.37,30106.44,42418.81,70.97,33.04
,7126.40,3411.81,14556.77,10283.14,10538.21,24839.91,35378.12,70.21,41.40
,7181.30,3761.50,13462.10,8963.90,10942.80,22426.00,33368.80,67.21,39.97
,7084.40,3575.60,14908.90,11340.20,10660.00,26249.10,36909.10,71.12,43.20


## 2. Load release dates and align report weeks


In [3]:
dates_path = ROOT / "data/season_release_dates.json"

release_dates = pd.read_json(dates_path)
release_dates["release_date"] = pd.to_datetime(
    release_dates["release_date"]
)

# 第二季及以后，且有明确上线日期
events = release_dates.loc[
    release_dates["season_number"].gt(1)
    & release_dates["release_date"].notna()
].copy()

# 对齐到上线所在报告周的周日
events["launch_week"] = (
    events["release_date"]
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

events = events.sort_values(["show_title", "season_number"])

display(events[[
    "show_title", "season_number",
    "release_date", "launch_week", "release_mode"
]])

show_title,season_number,release_date,launch_week,release_mode
Bridgerton,2,2022-03-25,2022-03-27,all_at_once
Bridgerton,3,2024-05-16,2024-05-19,split_batches
Bridgerton,4,2026-01-29,2026-02-01,split_batches
DAN DA DAN,2,2025-07-03,2025-07-06,weekly
Manifest,4,2022-11-04,2022-11-06,split_batches
Ms. Rachel,2,2025-09-01,2025-09-07,all_at_once
Squid Game,2,2024-12-26,2024-12-29,all_at_once
Squid Game,3,2025-06-27,2025-06-29,all_at_once
Stranger Things,2,2017-10-27,2017-10-29,all_at_once
Stranger Things,3,2019-07-04,2019-07-07,all_at_once


## 3. Check full chart coverage for all-at-once launches


In [4]:
# 检查每个类别、每周的榜单是否完整
coverage = tv.groupby(["category", "week"])["weekly_rank"].agg(
    rows="size",
    ranks_ok=lambda x: set(x) == set(range(1, 11))
)
complete_weeks = set(
    coverage.index[coverage["rows"].eq(10) & coverage["ranks_ok"]]
)

# 为上线事件匹配榜单类别
show_categories = tv[["show_title", "category"]].drop_duplicates()

all_at_once_events = (
    events.loc[events["release_mode"].eq("all_at_once")]
    .merge(show_categories, on="show_title", validate="many_to_one")
)

# 从第−12周到第+11周，共24周
all_at_once_events["full_24_weeks"] = all_at_once_events.apply(
    lambda e: all(
        (e["category"], e["launch_week"] + pd.Timedelta(weeks=r))
        in complete_weeks
        for r in range(-12, 12)
    ),
    axis=1
)

display(all_at_once_events[[
    "show_title", "season_number", "launch_week", "full_24_weeks"
]])

selected_events = all_at_once_events.loc[
    all_at_once_events["full_24_weeks"]
].copy()

print("Eligible events:", len(selected_events))

show_title,season_number,launch_week,full_24_weeks
Bridgerton,2,2022-03-27,True
Ms. Rachel,2,2025-09-07,True
Squid Game,2,2024-12-29,True
Squid Game,3,2025-06-29,True
Stranger Things,2,2017-10-29,False
Stranger Things,3,2019-07-07,False
The Gentlemen,2,2026-09-06,False


Eligible events: 4


## 4. Identify eligible older seasons


In [5]:
# 提取榜单中的季数；无法识别的季名保留为空
tv["season_number"] = pd.to_numeric(
    tv["season_title"].str.extract(
        r"(?:Season|Part)\s+(\d+)", expand=False
    ),
    errors="coerce"
).astype("Int64")

season_catalog = tv[[
    "category", "show_title", "season_title", "season_number"
]].drop_duplicates()

# 将每个上线事件与同剧、同类别的季匹配
event_old_seasons = selected_events[[
    "category", "show_title", "season_number", "launch_week"
]].rename(columns={"season_number": "new_season_number"}).merge(
    season_catalog.rename(columns={
        "season_title": "old_season_title",
        "season_number": "old_season_number"
    }),
    on=["category", "show_title"],
    how="inner"
)

event_old_seasons = event_old_seasons.loc[
    event_old_seasons["old_season_number"]
    < event_old_seasons["new_season_number"]
].sort_values(["show_title", "new_season_number", "old_season_number"])

display(event_old_seasons[[
    "show_title", "new_season_number",
    "old_season_title", "old_season_number", "launch_week"
]])

show_title,new_season_number,old_season_title,old_season_number,launch_week
Bridgerton,2,Bridgerton: Season 1,1,2022-03-27
Ms. Rachel,2,Ms. Rachel: Season 1,1,2025-09-07
Squid Game,2,Squid Game: Season 1,1,2024-12-29
Squid Game,3,Squid Game: Season 1,1,2025-06-29
Squid Game,3,Squid Game: Season 2,2,2025-06-29


## 5. Build the 24-week all-at-once panel


In [6]:
# 每个“上线事件—旧季”组合展开为24周
relative_weeks = pd.DataFrame({"relative_week": range(-12, 12)})

weekly_panel = event_old_seasons.merge(relative_weeks, how="cross")
weekly_panel["week"] = (
    weekly_panel["launch_week"]
    + pd.to_timedelta(weekly_panel["relative_week"] * 7, unit="D")
)

# 原始榜单中实际出现的剧季及周
chart_keys = ["category", "show_title", "old_season_title", "week"]

observed = (
    tv[["category", "show_title", "season_title", "week"]]
    .rename(columns={"season_title": "old_season_title"})
    .drop_duplicates()
    .assign(in_top10=1)
)

weekly_panel = weekly_panel.merge(
    observed, on=chart_keys, how="left", validate="many_to_one"
)
weekly_panel["in_top10"] = (
    weekly_panel["in_top10"].fillna(0).astype(int)
)

# 查看每个旧季在−12to+11周的上榜情况
weekly_matrix = weekly_panel.pivot(
    index=["show_title", "new_season_number", "old_season_title"],
    columns="relative_week",
    values="in_top10"
)

display(weekly_matrix)
print("Weekly panel rows:", len(weekly_panel))

relative_week,-12,-11,-10,-9,-8,-7,-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6,7,8,9,10,11
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,0,0
,0,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0


Weekly panel rows: 120


## 6. Export event-level chart counts


In [7]:
# 按事件统计每周上榜旧季数
chart_data = weekly_panel.groupby(
    ["relative_week", "show_title", "new_season_number"]
)["in_top10"].sum().unstack(["show_title", "new_season_number"])

# 简化列名
chart_data.columns = [
    f"{title} S{season}" for title, season in chart_data.columns
]
chart_data = chart_data.reindex(range(-12, 12))
chart_data.index.name = "relative_week"
chart_data = chart_data.reset_index()

display(chart_data)

chart_data.to_csv(
    OUT / "all_at_once_24week_chart_data.csv",
    index=False,
    encoding="utf-8-sig"
)

relative_week,Bridgerton S2,Ms. Rachel S2,Squid Game S2,Squid Game S3
-12,0,0,0,0
-11,0,0,0,0
-10,0,1,0,0
-9,0,1,0,0
-8,0,1,0,0
-7,0,1,0,0
-6,0,1,0,0
-5,0,1,0,0
-4,0,1,0,0
-3,0,1,0,0


## 7. Summarize all-at-once weekly counts


In [8]:
total_weekly = (
    weekly_panel.groupby("relative_week", as_index=False)
    .agg(total_old_seasons_in_top10=("in_top10", "sum"))
    .sort_values("relative_week")
)

display(total_weekly)

total_weekly.to_csv(
    OUT / "all_at_once_24week_total.csv",
    index=False,
    encoding="utf-8-sig"
)

relative_week,total_old_seasons_in_top10
-12,0
-11,0
-10,1
-9,1
-8,1
-7,1
-6,1
-5,1
-4,1
-3,1


## 8. Select split-batch launches with full coverage


In [9]:
split_events = (
    events.loc[events["release_mode"].eq("split_batches")]
    .merge(show_categories, on="show_title", validate="many_to_one")
)

split_events["full_24_weeks"] = split_events.apply(
    lambda e: all(
        (e["category"], e["launch_week"] + pd.Timedelta(weeks=r))
        in complete_weeks
        for r in range(-12, 12)
    ),
    axis=1
)

display(split_events[[
    "show_title", "season_number", "launch_week", "full_24_weeks"
]])

selected_split_events = split_events.loc[
    split_events["full_24_weeks"]
].copy()

print("Eligible events:", len(selected_split_events))

show_title,season_number,launch_week,full_24_weeks
Bridgerton,3,2024-05-19,True
Bridgerton,4,2026-02-01,True
Manifest,4,2022-11-06,True
Stranger Things,4,2022-05-29,True
Stranger Things,5,2025-11-30,True
Wednesday,2,2025-08-10,True


Eligible events: 6


## 9. Harmonize season names for the split sample


In [10]:
# 在副本中统一 Stranger Things 第一季的季名
split_tv = tv.copy()

mask = split_tv["show_title"].eq("Stranger Things") & (
    split_tv["season_title"].isna()
    | split_tv["season_title"].eq("Stranger Things 1")
)
split_tv.loc[mask, "season_title"] = "Stranger Things: Season 1"

# 识别标准季名及 Stranger Things 的特殊季名格式
season_num = split_tv["season_title"].str.extract(
    r"(?:Season|Part)\s+(\d+)", expand=False
)
special_num = split_tv["season_title"].str.extract(
    r"^Stranger Things\s+(\d+)$", expand=False
)
split_tv["season_number"] = pd.to_numeric(
    season_num.fillna(special_num), errors="coerce"
).astype("Int64")

split_catalog = split_tv[[
    "category", "show_title", "season_title", "season_number"
]].drop_duplicates()

split_old_seasons = selected_split_events[[
    "category", "show_title", "season_number", "launch_week"
]].rename(columns={"season_number": "new_season_number"}).merge(
    split_catalog.rename(columns={
        "season_title": "old_season_title",
        "season_number": "old_season_number"
    }),
    on=["category", "show_title"],
    how="inner"
)

split_old_seasons = split_old_seasons.loc[
    split_old_seasons["old_season_number"]
    < split_old_seasons["new_season_number"]
].sort_values(["show_title", "new_season_number", "old_season_number"])

display(split_old_seasons[[
    "show_title", "new_season_number",
    "old_season_title", "old_season_number"
]])

show_title,new_season_number,old_season_title,old_season_number
Bridgerton,3,Bridgerton: Season 1,1
Bridgerton,3,Bridgerton: Season 2,2
Bridgerton,4,Bridgerton: Season 1,1
Bridgerton,4,Bridgerton: Season 2,2
Bridgerton,4,Bridgerton: Season 3,3
Manifest,4,Manifest: Season 1,1
Manifest,4,Manifest: Season 2,2
Manifest,4,Manifest: Season 3,3
Stranger Things,4,Stranger Things: Season 1,1
Stranger Things,4,Stranger Things 2,2


## 10. Build the split-batch weekly panel


In [11]:
split_weekly_panel = split_old_seasons.merge(
    pd.DataFrame({"relative_week": range(-12, 12)}),
    how="cross"
)

split_weekly_panel["week"] = (
    split_weekly_panel["launch_week"]
    + pd.to_timedelta(split_weekly_panel["relative_week"] * 7, unit="D")
)

split_observed = (
    split_tv[["category", "show_title", "season_title", "week"]]
    .rename(columns={"season_title": "old_season_title"})
    .drop_duplicates()
    .assign(in_top10=1)
)

split_weekly_panel = split_weekly_panel.merge(
    split_observed,
    on=["category", "show_title", "old_season_title", "week"],
    how="left",
    validate="many_to_one"
)

split_weekly_panel["in_top10"] = (
    split_weekly_panel["in_top10"].fillna(0).astype(int)
)

split_weekly_matrix = split_weekly_panel.pivot(
    index=["show_title", "new_season_number", "old_season_title"],
    columns="relative_week",
    values="in_top10"
)

display(split_weekly_matrix)
print("Weekly panel rows:", len(split_weekly_panel))

relative_week,-12,-11,-10,-9,-8,-7,-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6,7,8,9,10,11
,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0
,1,1,1,1,1,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0
,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,0


Weekly panel rows: 384


## 11. Summarize split-batch weekly counts


In [12]:
split_total_weekly = (
    split_weekly_panel.groupby("relative_week", as_index=False)
    .agg(total_old_seasons_in_top10=("in_top10", "sum"))
    .sort_values("relative_week")
)

display(split_total_weekly)

split_total_weekly.to_csv(
    OUT / "split_batches_24week_total.csv",
    index=False,
    encoding="utf-8-sig"
)

relative_week,total_old_seasons_in_top10
-12,3
-11,2
-10,1
-9,2
-8,1
-7,0
-6,0
-5,0
-4,1
-3,1


## 12. Label previous and earlier seasons


In [13]:
def label_old_seasons(panel):
    result = panel.copy()
    result["old_season_group"] = "earlier_seasons"

    is_previous = (
        result["old_season_number"]
        == result["new_season_number"] - 1
    )
    result.loc[is_previous, "old_season_group"] = "previous_season"
    return result

comparison_all = label_old_seasons(weekly_panel)
comparison_split = label_old_seasons(split_weekly_panel)

# 检查每个事件中，两组各有多少个旧季
for name, panel in [
    ("all_at_once", comparison_all),
    ("split_batches", comparison_split)
]:
    print(name)
    display(
        panel.groupby([
            "show_title", "new_season_number", "old_season_group"
        ])["old_season_number"]
        .nunique()
        .unstack(fill_value=0)
    )

all_at_once


old_season_group,earlier_seasons,previous_season
,0,1
,0,1
,0,1
,1,1


split_batches


old_season_group,earlier_seasons,previous_season
,1,1
,2,1
,2,1
,2,1
,3,1
,0,1


## 13. Require both season groups within each event


In [14]:
def keep_comparable_events(panel):
    event_keys = ["show_title", "new_season_number"]

    group_counts = (
        panel.groupby(event_keys)["old_season_group"]
        .nunique()
        .reset_index(name="group_count")
    )

    eligible = group_counts.loc[
        group_counts["group_count"].eq(2), event_keys
    ]

    return panel.merge(
        eligible, on=event_keys, how="inner", validate="many_to_one"
    )

paired_all = keep_comparable_events(comparison_all)
paired_split = keep_comparable_events(comparison_split)

for name, panel in [
    ("all_at_once", paired_all),
    ("split_batches", paired_split)
]:
    print(name)
    display(
        panel[["show_title", "new_season_number"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

all_at_once


show_title,new_season_number
Squid Game,3


split_batches


show_title,new_season_number
Bridgerton,3
Bridgerton,4
Manifest,4
Stranger Things,4
Stranger Things,5


## 14. Calculate per-season weekly chart shares


In [15]:
def calculate_group_rates(panel):
    return (
        panel.groupby([
            "show_title", "new_season_number",
            "relative_week", "old_season_group"
        ], as_index=False)
        .agg(
            old_seasons=("old_season_number", "nunique"),
            seasons_in_top10=("in_top10", "sum"),
            top10_rate=("in_top10", "mean")
        )
    )

rates_all = calculate_group_rates(paired_all)
rates_split = calculate_group_rates(paired_split)

display(rates_all.head(8))
display(rates_split.head(8))

show_title,new_season_number,relative_week,old_season_group,old_seasons,seasons_in_top10,top10_rate
Squid Game,3,-12,earlier_seasons,1,0,0.0
Squid Game,3,-12,previous_season,1,0,0.0
Squid Game,3,-11,earlier_seasons,1,0,0.0
Squid Game,3,-11,previous_season,1,0,0.0
Squid Game,3,-10,earlier_seasons,1,0,0.0
Squid Game,3,-10,previous_season,1,0,0.0
Squid Game,3,-9,earlier_seasons,1,0,0.0
Squid Game,3,-9,previous_season,1,0,0.0


show_title,new_season_number,relative_week,old_season_group,old_seasons,seasons_in_top10,top10_rate
Bridgerton,3,-12,earlier_seasons,1,0,0.0
Bridgerton,3,-12,previous_season,1,0,0.0
Bridgerton,3,-11,earlier_seasons,1,0,0.0
Bridgerton,3,-11,previous_season,1,0,0.0
Bridgerton,3,-10,earlier_seasons,1,0,0.0
Bridgerton,3,-10,previous_season,1,0,0.0
Bridgerton,3,-9,earlier_seasons,1,0,0.0
Bridgerton,3,-9,previous_season,1,0,0.0


## 15. Compare cumulative chart weeks with equal event weighting


In [16]:
def count_top10_weeks(panel):
    return (
        panel.groupby([
            "show_title", "new_season_number",
            "old_season_group", "old_season_number"
        ], as_index=False)
        .agg(top10_weeks=("in_top10", "sum"))
    )

weeks_all = count_top10_weeks(paired_all)
weeks_split = count_top10_weeks(paired_split)

def summarize_weeks(season_weeks):
    # 先在每个事件内，计算两组平均每季上榜周数
    event_weeks = (
        season_weeks.groupby([
            "show_title", "new_season_number", "old_season_group"
        ])["top10_weeks"].mean()
    )

    # 再对事件等权平均
    return (
        event_weeks.groupby("old_season_group")
        .mean()
        .reindex(["previous_season", "earlier_seasons"])
        .rename("mean_top10_weeks")
        .reset_index()
    )

weeks_summary_all = summarize_weeks(weeks_all)
weeks_summary_split = summarize_weeks(weeks_split)

print("all_at_once")
display(weeks_summary_all)

print("split_batches")
display(weeks_summary_split)

all_at_once


old_season_group,mean_top10_weeks
previous_season,4.0
earlier_seasons,3.0


split_batches


old_season_group,mean_top10_weeks
previous_season,7.000000
earlier_seasons,7.366667


## 16. Export results and create figures


In [17]:
# Export the selected sample, weekly panel, and normalized supplementary comparison.
import sys
sys.path.insert(0, str(ROOT / 'src'))
from plot_svg import line_panels, bars, COLORS

event_rows = []
for mode, panel in [('all_at_once',weekly_panel),('split_batches',split_weekly_panel)]:
    for (show,season), part in panel.groupby(['show_title','new_season_number']):
        event_rows.append({'release_mode':mode,'show_title':show,'new_season_number':int(season),'older_seasons':part.old_season_number.nunique()})
event_sample=pd.DataFrame(event_rows)
event_sample.to_csv(OUT/'event_sample.csv',index=False)
for name,table in [('all_at_once_panel',weekly_panel),('split_batches_panel',split_weekly_panel),
                   ('all_at_once_previous_vs_earlier',weeks_summary_all),('split_batches_previous_vs_earlier',weeks_summary_split)]:
    table.to_csv(OUT/f'{name}.csv',index=False)

for expected_name,table in [('all_at_once_24week_total',total_weekly),('split_batches_24week_total',split_total_weekly)]:
    assert table.relative_week.tolist()==list(range(-12,12))
    assert table.total_old_seasons_in_top10.ge(0).all()

line_panels(FIGURES/'weekly_chart_patterns.svg',{
    f'All at once: {len(selected_events)} launches, {len(event_old_seasons)} event-season pairs':list(total_weekly.itertuples(index=False,name=None)),
    f'Split batches: {len(selected_split_events)} launches, {len(split_old_seasons)} event-season pairs':list(split_total_weekly.itertuples(index=False,name=None)),
},'Older-season chart appearances around a new-season launch','24 report weeks; counts are descriptive and depend on the selected sample')

rows=[]
for i,(mode,table) in enumerate([('All at once',weeks_summary_all),('Split batches',weeks_summary_split)]):
    for r in table.itertuples(index=False):
        rows.append((f"{mode} | {r.old_season_group.replace('_',' ')}",r.mean_top10_weeks,COLORS[i]))
bars(FIGURES/'previous_vs_earlier.svg',rows,'Previous seasons do not consistently lead earlier seasons',
     'Equal event weighting; 1 eligible all-at-once event and 5 split-batch events','Mean chart weeks per season within a 24-week window')
print('Selected sample:',len(event_sample),'launches across',event_sample.show_title.nunique(),'series')
display(event_sample)


Selected sample: 10 launches across 6 series


release_mode,show_title,new_season_number,older_seasons
all_at_once,Bridgerton,2,1
all_at_once,Ms. Rachel,2,1
all_at_once,Squid Game,2,1
all_at_once,Squid Game,3,2
split_batches,Bridgerton,3,2
split_batches,Bridgerton,4,3
split_batches,Manifest,4,3
split_batches,Stranger Things,4,3
split_batches,Stranger Things,5,4
split_batches,Wednesday,2,1
